# センサーの最適配置（元の課題）: Amplify の QUBO と古典的手法の比較

国際航業「センサーの最適配置」を、次の手法で解いて比べます。式は `placement/README.md` にあります。

| 手法 | 中身 |
|---|---|
| MILP（古典） | HiGHS による厳密解。最適であることの証明付き |
| 貪欲法（古典） | コスト当たりの不足の減り方が最大の手を繰り返す |
| QUBO 1 | 課題をそのまま Amplify の制約（`greater_equal` / `less_equal`）で書いたもの。スラック変数あり |
| QUBO 2 | ラグランジュ緩和（スラック変数なし）。乗数を外側のループで調整 |
| ＋修復 | QUBO の解の不足を貪欲法の手で補い、要らないセンサーを外したもの |

**Amplify のトークンを入れると QUBO を Fixstars Amplify AE で**解きます。AE は制約の重みを自動で調整するため、QUBO 1 でこちらが設定した重みは使われません。空欄なら、同じ QUBO をローカルの焼きなまし（neal）で解きます。

## 1. 準備

In [ ]:
!pip install -q amplify dimod dwave-neal
!apt-get -qq install -y fonts-ipafont-gothic > /dev/null
!rm -rf Q-Questgrow- && git clone -q -b claude/request-5v4fsc https://github.com/mori560269/Q-Questgrow-.git
%cd Q-Questgrow-/placement
import matplotlib.font_manager as fm, matplotlib.pyplot as plt
try:
    fm.fontManager.addfont('/usr/share/fonts/opentype/ipafont-gothic/ipag.ttf')
    plt.rcParams['font.family'] = ['IPAGothic', 'DejaVu Sans']
except Exception:
    pass
import sys, json, numpy as np, pandas as pd
sys.path.insert(0, '.')

## 2. 設定（ここだけ変えれば条件を変えられます）

In [ ]:
from getpass import getpass
# トークンは Colab の「シークレット」(左の鍵アイコン) に AMPLIFY_TOKEN として登録しておくのが安全です.
# 登録が無ければ入力欄が出ます (空のまま Enter でローカルの焼きなまし).
token = None
try:
    from google.colab import userdata
    token = userdata.get('AMPLIFY_TOKEN')
except Exception:
    pass
token = (token or getpass('Amplify のアクセストークン (空のまま Enter でローカルの焼きなまし) ')).strip() or None
backend = 'amplify' if token else 'neal'

SEEDS = 3            # QUBO を何回解くか (乱数の種)
Q1_TIME_MS = 10000   # QUBO 1: AE の制限時間 [ms] / 1 回
Q1_SWEEPS = 8000     # QUBO 1: neal のスイープ数
Q2_ITERS = 100       # QUBO 2: 乗数の外側ループの回数
Q2_TIME_MS = 500     # QUBO 2: AE の制限時間 [ms] / 内側 1 回
Q2_SWEEPS = 200      # QUBO 2: neal のスイープ数 / 内側 1 回
if backend == 'amplify':
    Q2_ITERS, Q2_TIME_MS = 50, 1000   # AE は 1 回 500 ms だと収束前に時間切れになるので 1 秒に (反復は 50 回)
# AE の警告は qubo_amplify.py の中で処理します (重みの警告は表示しない. 時間切れは数えて, QUBO 2 は制限時間を自動で延ばす)
print('QUBO の解き方:', 'Fixstars Amplify AE' if backend == 'amplify' else 'ローカルの焼きなまし (neal)')
# AE につながるかを最初に確かめる (トークンの誤り・通信の不具合はここで分かる)
if backend == 'amplify':
    from qubo_amplify import check_ae
    try:
        print('AE 接続テスト OK:', check_ae(token))
    except Exception as e:
        print('AE に接続できません:', e)
        print('→ トークンを確かめてもう一度このセルを実行するか, 空欄にしてローカルの焼きなましで続けてください')
        raise


## 2.5 QUBO 行列を見る

QUBO は `E(x) = Σ_i Q_ii x_i + Σ_{i<j} Q_ij x_i x_j + 定数` の形です（x_i は 0/1）。x_i² = x_i なので、1 次の項は対角に置いています。
ここでは、実際に解いている行列を取り出して表示し、検算します。トークンは不要です。

### QUBO 1（Amplify の制約をそのまま展開した行列）

Amplify が `greater_equal` / `less_equal` をスラック変数付きの 2 乗ペナルティに展開した後の行列です。
変数を **x（センサー）→ y（マスの被覆）→ スラック** の順に並べています（上三角だけを表示）。

In [ ]:
import scipy.sparse as sp
from common import Problem
import qubo_view as qv

Q1, const1, blk = qv.qubo1_matrix(Problem(cell_m=2000.0))
nx, ny, ns = blk['x'], blk['y'], blk['slack']
print(f'QUBO 1: 変数 {Q1.shape[0]:,} 個 (x {nx}, y {ny:,}, スラック {ns:,}) / 0 でない係数 {Q1.nnz:,} 個 / 定数 {const1:.4g}')
edges = [0, nx, nx + ny, nx + ny + ns]; names = ['x', 'y', 'スラック']
cnt = pd.DataFrame([[Q1[edges[a]:edges[a+1], edges[b]:edges[b+1]].nnz for b in range(3)] for a in range(3)],
                   index=names, columns=names)
print('ブロックごとの 0 でない係数の数 (行 × 列)'); display(cnt)
fig, ax = plt.subplots(figsize=(6.5, 6.5))
ax.spy(Q1, markersize=0.03, color='k')
for e in edges[1:-1]:
    ax.axhline(e, color='tab:red', lw=1.5, zorder=3); ax.axvline(e, color='tab:red', lw=1.5, zorder=3)
ax.set_title(f'QUBO 1 の行列 (黒 = 0 でない係数)\n赤線 = 境目: x は 0〜{nx}, y は {nx}〜{nx + ny}, 残りがスラック')
plt.show()

- スラック変数が全体の約 8 割を占めます。不等式 1 本ごとに、余りを 2 進数で表す変数が足されるためです。
- 係数は 1,000 万を超える大きさになります（重みつきの 2 乗ペナルティのため）。焼きなましが動きにくい原因の一つです。
- Amplify AE に送るときは、AE が重みを自分で調整するので、この行列の重みとは値が変わります。

### QUBO 2（ラグランジュ緩和の行列、変数 258 個）

`E(x) = Σ c_i x_i − Σ_m μ_m (Σ A_mi x_i − Σ B_mij x_i x_j) − Σ ν_m α_i x_i − Σ ρ_m β_i x_i + P Σ x_{p,k} x_{p,k'}`

Amplify AE には、この行列（1 次の係数と 2 次の係数）を `gen.matrix` でそのまま送ります。乗数は外側のループの初期値（μ = 0.5、ν = 30、ρ = 15）です。

In [ ]:
from matplotlib.colors import SymLogNorm
P1 = Problem(cell_m=1000.0)
lin2, Q2, terms2, mults2 = qv.qubo2_matrix(P1)
full = Q2 + np.diag(lin2)
print(f'QUBO 2: 変数 {Q2.shape[0]} 個 / 0 でない係数 {int((full != 0).sum()):,} 個')
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(full, cmap='RdBu_r', norm=SymLogNorm(linthresh=1, vmin=-abs(full).max(), vmax=abs(full).max()))
plt.colorbar(im, ax=ax, label='係数 (対数目盛り)')
ax.set_title('QUBO 2 の行列\n対角 = 1 次の係数 / 青 = 置くと得, 赤 = 置くと損')
ax.set_xlabel('変数 i = (候補地, 種類)'); ax.set_ylabel('変数 j')
plt.show()

# 最初の 3 候補地 (9 変数) の部分を表で見る
lab = qv.labels(P1)
part = pd.DataFrame(full[:9, :9].round(1), index=lab[:9], columns=lab[:9])
print('最初の 3 候補地の部分 (対角 = 1 次の係数, 右上 = 2 次の係数)'); display(part)

- 対角（1 次の係数）は「コスト − 覆う面積・α・β へのごほうび」で、マイナスなら置くと得です。
- 同じ候補地の 3 種類どうし（表の 3×3 の右上）は大きなプラスで、1 候補地に 2 台置くと損になります。
- 円が重なる 2 台どうしは、重なったマスの分だけプラスになります（同じ場所を 2 回数えないため）。

### 検算: 行列で計算したエネルギー ＝ 式の項を別々に足したもの

MILP の最適解（200 m マス）を入れて、2 通りでエネルギーを計算します。
あわせて、面積の項（ボンフェローニの下界 ΣA − ΣB）が本当の被覆マス数を超えていないか（多く見積もっていないか）も確かめます。

In [ ]:
from classical import solve_milp
x_opt, _ = solve_milp(Problem(cell_m=200.0))
chk = qv.check_qubo2(P1, x_opt, lin2, Q2, terms2, mults2)
print('行列で計算したエネルギー  ', chk['energy_from_matrix'])
print('式の項を別々に足したもの  ', chk['energy_from_terms'])
print('一致:', chk['match'], ' / 総コスト', chk['cost'])
display(pd.DataFrame(dict(市町村=P1.names, 下界_ΣA_minus_ΣB=chk['bonferroni_lower_bound'],
                          本当の被覆マス数=chk['true_union'], マス数=P1.Nm)))
print('下界が本当の値を超えていない:', chk['bound_ok'])

- 2 つのエネルギーが一致すれば、行列が式どおりに作られていることになります。
- センサーが密な市町村では、下界がマイナスになるほど小さくなります（3 台以上の重なりを引きすぎるため）。
  面積を少なめに見積もるので「覆えたつもり」にはなりませんが、その分センサーを多めに置く方向に偏ります。QUBO 2 の解が最適より高くなる原因の一つです。

## 3. 全手法の比較（コスト・時間・実行可能か）

In [ ]:
from experiments import run_all
prob, rows = run_all(backend, token, seeds=SEEDS, q1_sweeps=Q1_SWEEPS, q1_time_ms=Q1_TIME_MS,
                     q2_iters=Q2_ITERS, q2_sweeps=Q2_SWEEPS, q2_time_ms=Q2_TIME_MS)
table = pd.DataFrame([{k: v for k, v in r.items() if k != '_x'} for r in rows])
if backend == 'amplify':
    from qubo_amplify import AE_STATS
    print('AE の呼び出し', AE_STATS)
table

配置の地図（実行可能な解だけ）

In [ ]:
from common import save_map
from IPython.display import Image
maps = {}
for r in rows:
    if r.get('_x') is not None and r['実行可能']:
        maps[r['手法']] = prob.evaluate(np.array(r['_x']), cell_m=1000)
save_map(prob, maps, 'placement_maps.png')
Image('placement_maps.png')

## 4. 同じ時間でどこまで良い解が出るか

QUBO 2 は、反復回数を時間の予算に合わせて変えます。MILP は時間制限を変えます。
時間切れのときは、それまでに見つけた解を使います。

In [ ]:
from experiments import equal_time
et = pd.DataFrame(equal_time(backend, token, budgets=(1, 3, 10)))
et

## 5. 規模を変えた比較（候補地を増やす）

元の 86 か所に、格子状の候補地を足して問題を大きくします。
- `8000`: 8 km 間隔、184 か所
- `5000`: 5 km 間隔、342 か所
- `3500`: 3.5 km 間隔、611 か所

ローカルの焼きなましでは 611 か所に数分かかります。

In [ ]:
from experiments import scaling
sc = pd.DataFrame(scaling(backend, token, grids=(None, 8000, 5000), q2_iters=Q2_ITERS,
                          q2_sweeps=Q2_SWEEPS, q2_time_ms=Q2_TIME_MS))
sc

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot(sc['候補地'], sc['MILP_秒'], 'o-', label='MILP')
ax[0].plot(sc['候補地'], sc['QUBO2修復_秒'], 's-', label='QUBO 2 + 修復')
ax[0].set_xlabel('候補地の数'); ax[0].set_ylabel('時間 [秒]'); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(sc['候補地'], sc['MILP_コスト'], 'o-', label='MILP (最適)')
ax[1].plot(sc['候補地'], sc['QUBO2修復_コスト'], 's-', label='QUBO 2 + 修復')
ax[1].set_xlabel('候補地の数'); ax[1].set_ylabel('総調達コスト'); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

## 6. 結果を保存

In [ ]:
from google.colab import files
out = dict(backend=backend, settings=dict(SEEDS=SEEDS, Q1_TIME_MS=Q1_TIME_MS, Q1_SWEEPS=Q1_SWEEPS,
           Q2_ITERS=Q2_ITERS, Q2_TIME_MS=Q2_TIME_MS, Q2_SWEEPS=Q2_SWEEPS),
           compare=table.to_dict('records'), equal_time=et.to_dict('records'), scaling=sc.to_dict('records'))
json.dump(out, open(f'placement_results_{backend}.json', 'w'), ensure_ascii=False, indent=1, default=str)
files.download(f'placement_results_{backend}.json')
files.download('placement_maps.png')